In [60]:
import pandas as pd
import pytz
import re

# many sheets
anySheet = pd.ExcelFile("chatbot_phase2.xlsx") # ['ยังไม่แยกอาหาร this', 'แยกส่วนประกอบมาแล้ว', 'Total']
sheetName = anySheet.sheet_names

# get a specific sheet
df = pd.read_excel("chatbot_phase2.xlsx", sheet_name=sheetName[1])


In [62]:
def clean_menu(text):
    text = str(text)
    # Replace newline with space
    text = text.replace("\n", " ")
    # Replace punctuation with space (like . , ; )
    text = re.sub(r"[.,;]", " ", text)
    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    # Split by space
    items = text.split(" ")
    # Remove empty strings
    items = [i for i in items if i.strip()]
    return items

# Apply cleaning function
df["Cleaned_menu"] = df["List menu"].apply(clean_menu)

print(df["Cleaned_menu"])



0       [มะระ, หมู, ผักชี, เห็นชอบ, กะปิ, สะตอ, ข้าว, ...
1             [ข้าว, หมู, คะน้า, หอมใหญ่, มะเขือเทศ, ไข่]
2                                 [กาแฟ, ข้าวเหนียว, หมู]
3        [ข้าว, เนื้อวัว, ใบกะเพรา, พริก, กะเพรา, ไข่ไก่]
4       [ข้าว, เนื้อหมู, กะทิ, มะเขือพวง, ใบมะกรูด, ผั...
                              ...                        
1803    [ข้าว, แพนง, หมู, ไก่, เห็ด, ขิง, แอปเปิ้ล, มะ...
1804                [ก๋วยจั๊บ, หมูกรอบ, ตับ, หัวใจ, ม้าม]
1805                   [หมี่เหลือง, ไก่, ไอติม, กล้วยหอม]
1806    [เอ็กเบเนดิก, ไข่, เห็ด, มะเขือเทศ, อโวคาโด, ซ...
1807    [ปลาสำลีทอด, ผักกูดลวก, กระเจี๊ยบเขียว, มะเขือ...
Name: Cleaned_menu, Length: 1808, dtype: object


In [63]:
df1 = df.iloc[0:25]

In [64]:
import pandas as pd
from dotenv import load_dotenv
import os
from google import genai
from google.genai import types
import re, json
import json, re, unicodedata, difflib
import pandas as pd

load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

all_lists = "\n".join(str(lst) for lst in df1["List menu"])

prompt = f"""
จากลิสต์ต่อไปนี้แต่ละบรรทัด:
{all_lists}

1. แก้คำผิดในแต่ละคำ

2. ถ้าในลิสต์คำมีหลายคำอาหารหรือส่วนประกอบอยู่ร่วมกัน  
    ให้พิจารณาบริบทรอบข้างว่า คำเหล่านั้นสามารถรวมกันเป็นชื่ออาหารเดียวได้หรือไม่

    เช่น:
    ['น้ำเต้าหู้', 'muesli', 'ขนมปัง', 'multi', 'grain']  
    → ให้ตีความว่า "multi" + "grain" + "ขนมปัง" สามารถรวมเป็น "ขนมปังมัลติเกรน"  
    ผลลัพธ์จึงควรเป็น:  
    ["น้ำเต้าหู้", "ขนมปังมัลติเกรน"]

    **หมายเหตุ:**  
    - ให้พิจารณาการรวมคำโดยดูความสัมพันธ์ของคำอาหาร เช่น  
    คำภาษาอังกฤษที่บ่งบอกชนิด (เช่น “multi”, “grain”, “wholewheat”)  
    เมื่ออยู่ใกล้คำหลัก เช่น “ขนมปัง” ให้รวมเป็นคำเดียว  
    - ห้ามรวมคำที่ไม่เกี่ยวข้องกัน เช่น “น้ำเต้าหู้” กับ “ขนมปัง” ต้องแยกเป็นคนละเมนู

3. ถ้าเจอคำว่า "เนื้อ", "ไก่", "หมู", "ปลา", "ไข่", ผักเคล  
    - ให้ถือเป็น **ชื่ออาหารหลัก** และ **คงไว้ในผลลัพธ์เสมอ**  
    - ไม่ต้องลบหรือแยกเป็นส่วนประกอบ  
    - สามารถแก้คำสะกดผิดได้ตามปกติ  
    - แม้ว่าคำนี้ปรากฏเพียงลำพัง หรือไม่มีคำขยายอื่น ๆ ให้ยังเก็บคำนี้ไว้

4. ถ้าเจอคำภาษาอังกฤษ ให้ตรวจสอบว่าเป็นคำที่เกี่ยวข้องกับอาหารหรือไม่

    - ถ้าไม่เกี่ยวข้องกับอาหารหรือวัตถุดิบ ให้ลบคำนั้นออกจากรายการทั้งหมด  
    - ถ้าเป็นคำที่เกี่ยวข้องกับอาหารหรือวัตถุดิบ ให้แปลเป็นภาษาไทย และสะกดให้ถูกต้องตามมาตรฐาน

    ตัวอย่าง:
    - ["ข้าวกล้อง", "good", "taste"] → ["ข้าวกล้อง"]
    - ["bread", "egg"] → ["ขนมปัง", "ไข่"]
    - ["muesli", "breakfast", "set"] → ["มูสลี"]
    - ["tofu", "fresh"] → ["เต้าหู้"]
    - ["rice", "clean"] → ["ข้าว"]

    **หมายเหตุ:**  
    - ให้คงเฉพาะคำที่เป็นชื่ออาหารหรือวัตถุดิบจริง เช่น  
      "bread", "tofu", "muesli", "milk", "egg", "pasta", "rice", "chicken", "beef", "fish", "vegetable"  
    - คำภาษาอังกฤษทั่วไปที่เป็นคำขยายหรือคำเชิงพรรณนา เช่น  
      “good”, “best”, “fresh”, “healthy”, “fit”, “clean”, “menu”, “set”, “dish” ให้ลบทิ้งทั้งหมด  
    - การแปลชื่ออาหารเป็นไทยต้องสะกดให้ถูกต้อง เช่น  
      "bread" → "ขนมปัง", "tofu" → "เต้าหู้", "egg" → "ไข่", "rice" → "ข้าว", "muesli" → "มูสลี"
    - หากพบชื่อแบรนด์อาหารหรือเครื่องดื่ม เช่น "Ensure", "Ovaltine", "Nutella"  
    ให้แปลเป็นภาษาไทยและคงชื่อแบรนด์ไว้ เช่น  
        "Ensure" → "เอนชัวร์"  
        "Ovaltine" → "โอวัลติน"

5. ถ้าเจอชื่ออาหารที่ลงท้ายด้วยคำว่า "เจ" เช่น "กะเพราเจ", "ต้มยำเจ", "ผัดผักรวมเจ"  
    ให้คงคำว่า "เจ" ไว้ในชื่ออาหาร  
    - เช่น "กะเพราเจ" → "กะเพราเจ"  
    **หมายเหตุ:**  
    - ห้ามลบหรือเปลี่ยนคำว่า "เจ"
    
6. ลบคำที่ไม่ใช่ “เมนูอาหาร” หรือ “ส่วนประกอบของอาหาร” ออกทั้งหมด  
   เช่น คำที่เป็นคำพูดทั่วไปหรือคำผิดแปลกๆ ที่ไม่เกี่ยวกับอาหารโดยสิ้นเชิง เช่น  
   - "ชเย"  
   - "ไม่ได้ทาน"  
   - "ละกรรม"  
   - "นิดหน่อย"  
   - "อร่อยมาก"  
   - "ตัว"
   - หรือคำอื่นที่ไม่สื่อถึงชื่ออาหารหรือวัตถุดิบ  
   **ห้ามเหลือคำเหล่านี้ไว้ในผลลัพธ์เด็ดขาด**

7. รวมคำที่มีความหมายใกล้เคียงหรือซ้ำกันให้เหลือคำเดียว **อย่างสม่ำเสมอทั้งลิสต์**  
    - เช่น "ไข่ไก่" → "ไข่"  
    - "บรอกโคลี" → "ผักบรอกโคลี"  
    - "นมวัวโปรตีนรสช็อคโกแล็ต" → "นมวัวรสช็อกโกแลต"  
    - "ปั้นสิบ" → "ขนมปั้นสิบ"
    - "กะเพรา" → "ใบกะเพรา"   

ุ8. ถ้าพบชื่อพันธุ์ของผลไม้ ให้ตัดชื่อพันธุ์ออก เหลือเพียงชื่อผลไม้หลักและคำขยาย เช่น 
    **ยกเว้นกรณี “นมหนองโพ” ให้คงคำว่า “หนองโพ” ไว้**
    - "มะม่วงน้ำดอกไม้สุก" → "มะม่วงสุก"  
    - "ทุเรียนหมอนทอง" → "ทุเรียน"  
    - "กล้วยน้ำว้า" → "กล้วย"  
    - "เงาะโรงเรียน" → "เงาะ"
    - "นมหนองโพ" → "นมหนองโพ"  

9. ถ้าพบชื่ออาหารที่มีคำขยายหรือคำตกแต่งที่ไม่เกี่ยวข้องกับอาหาร เช่น “ชาววัง”, “สูตรโบราณ”, “ต้นตำรับ”, “สูตรคุณยาย”, “เยาวราช”, “เจ”, “มังสวิรัติ” ฯลฯ  
ให้ลบคำเหล่านั้นออก เหลือเฉพาะชื่ออาหารหลัก เช่น  
- "หมูทอดชาววัง" → "หมูทอด"  
- "ข้าวมันไก่เยาวราช" → "ข้าวมันไก่"  
- "แกงเขียวหวานสูตรโบราณ" → "แกงเขียวหวาน"

**เพิ่มเติม:**  
หลังจากลบคำขยายหรือคำตกแต่งเหล่านี้ออกแล้ว  
ให้ดำเนินการต่อโดยใช้กฎข้ออื่น ๆ ที่เกี่ยวข้องกับชื่ออาหารนั้นต่อไปตามลำดับ  
เช่น ถ้าผลลัพธ์เหลือเป็น “หมูทอด” ก็ให้แยกส่วนประกอบตามกฎของเมนูทอด (ข้อ 17)  
หรือถ้าเหลือเป็น “แกงเขียวหวาน” ก็ให้แยกตามกฎของเมนูแกง (ข้อ 12)


10. ถ้าพบชื่อเครื่องดื่มหรืออาหารที่ประกอบด้วยหลายส่วนผสมชัดเจน เช่น “ชาเขียวน้ำผึ้งมะนาว”  
ให้แยกออกเป็นรายการย่อยตามส่วนผสม เช่น  
- "ชาเขียวน้ำผึ้งมะนาว" → ["ชาเขียว", "น้ำผึ้ง", "น้ำมะนาว"]  
- "ชานมไข่มุกบราวน์ชูการ์" → ["ชานม", "ไข่มุก", "บราวน์ชูการ์"]

11. ถ้าเจอชื่ออาหารที่เป็นเมนูประเภท “เกาเหลา” เช่น
   - "เกาเหลาลูกชิ้นหมู"
   - "เกาเหลาลูกชิ้นเนื้อ" 
   
   ให้ทำดังนี้:  
   1. แยกชื่ออาหารหลัก “เกาเหลา” ออกเป็น **ส่วนประกอบหลักของซุป** เช่น  
      - "เกาเหลา" → ["น้ำซุป", "ผักชี", "ต้นหอม", "ผักอื่น ๆ ตามสูตร"]  
   2. รวมกับวัตถุดิบเฉพาะในชื่อ เช่น  
      - "ลูกชิ้นหมู"  
      - "ลูกชิ้นเนื้อ"  

   ตัวอย่าง:
   - "เกาเหลาลูกชิ้นหมู" → ["น้ำซุป", "ผักชี", "ต้นหอม", "ผักอื่น ๆ", "ลูกชิ้นหมู"]  
   - "เกาเหลาลูกชิ้นเนื้อ" → ["น้ำซุป", "ผักชี", "ต้นหอม", "ผักอื่น ๆ", "ลูกชิ้นเนื้อ"]  

   หมายเหตุ:  
   - ถ้าในชื่อมีวัตถุดิบอื่นเช่น "เกาเหลาหมูสับ" หรือ อินพุต มี ["เกาเหลาลูกชิ้นหมู", "หมูสับ"]  
     ให้รวมวัตถุดิบทั้งหมดเข้าใน list เดียว เช่น  
     - ["น้ำซุป", "ผักชี", "ต้นหอม", "ผักอื่น ๆ", "ลูกชิ้นหมู", "หมูสับ"]

12. ถ้าเจอชื่ออาหารที่ขึ้นต้นด้วยคำว่า “ผัด ...”  
    ให้แยกออกเป็น “วัตถุดิบหลักที่อยู่หลังคำว่าผัด” + “เครื่องปรุงพื้นฐานของเมนูผัด” เช่น  
    - "น้ำมัน", "กระเทียม", "ซีอิ๊ว", "น้ำปลา", "พริก", "น้ำตาล", "ไข่ (ถ้ามีในชื่อ)", "พริกไทย"

    ตัวอย่าง:
    - "ผัดขิงไก่" หรือ "ไก่ผัดขิง" → ["ไก่", "ขิง", "น้ำมัน", "กระเทียม", "ซีอิ๊ว", "น้ำปลา", "พริกไทย"]  
    - "ผัดคะน้าหมูกรอบ" → ["คะน้า", "หมูกรอบ", "น้ำมัน", "กระเทียม", "ซีอิ๊ว", "น้ำปลา", "พริก"]  
    - "ผัดกะเพราไก่ไข่ดาว" → ["ไก่", "กะเพรา", "พริก", "กระเทียม", "น้ำปลา", "น้ำตาล", "ไข่"]  
    - "ผัดพริกแกงหมู" → ["หมู", "พริกแกง", "น้ำมัน", "น้ำปลา", "น้ำตาล", "ใบมะกรูด"]  
    - "ผัดผักรวม" → ["ผักรวม", "น้ำมัน", "กระเทียม", "ซีอิ๊ว", "น้ำปลา", "พริกไทย"]

    หมายเหตุ:
    - ถ้ามีคำว่า “ผัดซีอิ๊ว”, “ผัดไทย”, “ผัดวุ้นเส้น” ให้ถือเป็นเมนูเฉพาะ  
      ให้เพิ่มส่วนประกอบของเส้น เช่น  
      - "ผัดซีอิ๊วหมู" → ["เส้นใหญ่", "หมู", "คะน้า", "ไข่", "ซีอิ๊วดำ", "น้ำปลา", "น้ำมัน"]  
      - "ผัดไทยกุ้งสด" → ["เส้นจันท์", "กุ้ง", "เต้าหู้", "ถั่วงอก", "ไข่", "น้ำมะขามเปียก", "น้ำปลา", "น้ำตาลปี๊บ"]

13. ถ้าเจอชื่ออาหารที่เป็น “แกง ...” เช่น  
    - "แกงหน่อไม้"  
    - "แกงเขียวหวานไก่"  
    - "แกงส้มชะอมกุ้ง"  
    - "แกงเลียงผักรวม"  
    ให้แยกออกเป็นรายการของ **ส่วนประกอบหลัก** ของแกงนั้น โดยใช้โครงสร้างพื้นฐานของเมนูนั้น ๆ เช่น  

    - "แกงหน่อไม้" → ["หน่อไม้", "น้ำใบย่านาง", "พริก", "ข่า", "ตะไคร้", "ใบมะกรูด", "ใบชะอม", "กะปิ", "น้ำปลา", "ปลาย่าง", "เกลือ"]  
    - "แกงเขียวหวาน" → ["พริกแกงเขียวหวาน", "กะทิ", "มะเขือ", "ใบโหระพา", "น้ำปลา", "น้ำตาลปี๊บ"]  
    - "แกงส้มชะอมกุ้ง" → ["ชะอม", "กุ้ง", "น้ำแกงส้ม", "น้ำมะขามเปียก", "พริก", "กระเทียม", "เกลือ"]  
    - "แกงเลียงผักรวม" → ["ผักรวม", "บวบ", "ฟักทอง", "เห็ด", "กะปิ", "พริก", "หอมแดง"]  

    ถ้าชื่อแกงมีวัตถุดิบระบุในชื่อ เช่น  
    - "แกงหน่อไม้ไก่" หรือ อินพุต เดิมมี ["แกงหน่อไม้", "ไก่"]  
    ให้รวมเป็นลิสต์ของส่วนประกอบหลักของ “แกงหน่อไม้” พร้อมวัตถุดิบเพิ่มเติมที่มีอยู่ เช่น  
    - ["หน่อไม้", "น้ำใบย่านาง", "พริก", "ข่า", "ตะไคร้", "ใบมะกรูด", "ใบชะอม", "กะปิ", "น้ำปลา", "ปลาย่าง", "เกลือ", "ไก่"]

14. ถ้าเจอชื่ออาหารที่เป็นเมนูข้าว เช่น "ข้าวหมาก", "ข้าวเหนียว", "ข้าวแช่" หรือเมนูข้าวอื่น ๆ  
    - ให้คงชื่ออาหารไว้ **ทั้งคำ**  
    - **ไม่ต้องแยกออกเป็นส่วนประกอบ**  
    - สามารถแก้คำสะกดผิดได้ตามปกติ  
    - **ยกเว้นเมนูข้าวต้ม** เช่น "ข้าวต้มหมู", "ข้าวต้มกุ้ง" ให้แยกตามกฎเมนูต้ม  
    - **ข้อสำคัญ:** เฉพาะเมนูข้าวจริง ๆ เท่านั้น เช่น “ข้าวหมาก” “ข้าวเหนียว”  
      - คำที่มี “ข้าว” แต่ไม่ใช่เมนูข้าว เช่น “ข้าวโพด” ให้ **ไม่ถือเป็นเมนูข้าว** และทำตามกฎทั่วไปแทน

    
15.  ถ้าเจอชื่ออาหารที่เป็น “...ต้ม” เช่น
    - "ข้าวโพดต้ม"
    - "ไข่ต้ม"
    - "ผักต้ม"
    - "ข้าวต้ม"
    - "ซุปผัก"
    
    ให้แยกเป็น **วัตถุดิบหลัก + ส่วนประกอบพื้นฐานของการต้ม** เช่น

    ตัวอย่าง:
    - "ข้าวโพดต้ม" → ["ข้าวโพด", "น้ำ", "เกลือ"]  
    - "ไข่ต้ม" → ["ไข่", "น้ำ", "เกลือ"]  
    - "ผักต้ม" → ["ผัก", "น้ำ", "เกลือ"]  
    - "ข้าวต้มหมู" → ["ข้าว", "หมู", "น้ำซุป", "เกลือ", "พริกไทย", "กระเทียม", "ต้นหอม"]  
    - "ซุปผัก" → ["ผัก", "น้ำซุป", "เกลือ", "พริกไทย", "สมุนไพรตามสูตร"]

    หมายเหตุ:
    - ถ้าเมนูต้มมีวัตถุดิบอื่นระบุในชื่อ เช่น "ไข่ต้มหมู", "ข้าวต้มกุ้ง"  
      ให้รวมวัตถุดิบเหล่านั้นเข้า list ด้วย เช่น  
      - "ไข่ต้มหมู" → ["ไข่", "หมู", "น้ำ", "เกลือ"]  
      - "ข้าวต้มกุ้ง" → ["ข้าว", "กุ้ง", "น้ำซุป", "เกลือ", "พริกไทย", "กระเทียม", "ต้นหอม"]

16. ถ้าเจอชื่ออาหารที่เป็น “เมนูผัด...” เช่น  
    - "ไก่ผัดขิง"  
    - "หมูผัดพริกไทยดำ"  
    - "เนื้อผัดน้ำมันหอย"  
    ให้แยกออกเป็นส่วนประกอบหลักของเมนูผัดพื้นฐาน + วัตถุดิบหลักในชื่อ เช่น  
    - "ไก่ผัดขิง" → ["ไก่", "ขิง", "น้ำมัน", "ซีอิ๊วขาว", "พริกไทย"]  
    - "หมูผัดพริกไทยดำ" → ["หมู", "พริกไทยดำ", "น้ำมัน", "ซอสปรุงรส", "น้ำปลา"]  
    - "เนื้อผัดน้ำมันหอย" → ["เนื้อวัว", "น้ำมันหอย", "พริก", "น้ำปลา", "น้ำตาล"]
    
17. ถ้าเจอชื่ออาหารที่เป็นเมนูประเภท “...ย่าง” เช่น
    - "ไก่ย่างพริกไทยดำ"
    - "ปลาย่างสมุนไพร"

    ให้แยกเป็น:
    1. วัตถุดิบหลัก (ตามชื่อเมนู) เช่น ไก่, ปลา
    2. เครื่องปรุงหรือเครื่องหมักสำหรับการย่าง เช่น พริกไทย, เกลือ, น้ำปลา, ซอสหมัก

ตัวอย่าง:
- "ไก่ย่างพริกไทยดำ" → ["ไก่", "พริกไทยดำ", "เกลือ", "น้ำปลา", "ซอสหมัก"]
- "ปลาย่างสมุนไพร" → ["ปลา", "สมุนไพร", "เกลือ", "น้ำมัน"]
    
18. ถ้าเจอชื่ออาหารที่เป็นเมนูทอด เช่น "...ทอด" หรือ "ไก่ทอด":
- แยกวัตถุดิบหลักจากชื่อเมนู
- ถ้ามีคำขยายหรือเครื่องปรุงเพิ่มเติมในชื่อ ให้รวมด้วย
- เพิ่มส่วนประกอบพื้นฐานของการทอด: แป้งทอดกรอบ, น้ำมันทอด, เกลือ, พริกไทย

ตัวอย่าง:
- "ไก่ทอด" → ["ไก่", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]
- "หมูทอดกระเทียม" → ["หมู", "กระเทียม", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]
- "ปลาทอด" → ["ปลา", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]
- "หมูทอด" → ["หมู", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]


19. ถ้าเจอชื่ออาหารที่เป็น “ส้มตำ...” หรือ “ยำ...”  
    ให้แยกเป็นวัตถุดิบหลัก + เครื่องปรุงยำพื้นฐาน เช่น  
    - "ส้มตำไทย" → ["มะละกอ", "มะเขือเทศ", "ถั่วฝักยาว", "ถั่วลิสง", "กระเทียม", "พริก", "น้ำปลา", "น้ำตาลปี๊บ", "น้ำมะนาว", "กุ้งแห้ง"]  
    - "ยำวุ้นเส้นหมูสับ" → ["วุ้นเส้น", "หมูสับ", "พริก", "หอมแดง", "น้ำปลา", "น้ำตาล", "น้ำมะนาว"]  

20. ถ้าเจอชื่ออาหารที่เป็น “ขนม...” หรือ “ของหวาน” เช่น  
    - "ขนมปังเนยย่าง"  
    - "สาคูไส้หมู"  
    ให้แยกเป็นวัตถุดิบหลักของของหวานนั้น เช่น  
    - "ขนมปังเนยย่าง" → ["ขนมปัง", "เนย", "น้ำตาล"]  
    - "สาคูไส้หมู" → ["สาคู", "หมู", "ถั่วลิสง", "กระเทียม", "พริกไทย", "น้ำตาลปี๊บ"]  

21. ถ้าเจอคำที่เกี่ยวข้องกับขนมปังชนิดต่างๆ เช่น "ขนมปัง" + "wholewheat", "multigrain", "มัลติเกรน"  
    ให้รวมเป็นชื่อขนมปังเดียว เช่น:  
    - "ขนมปัง" + "wholewheat" → "ขนมปังโฮลวีต" → ["แป้งโฮลวีต", "ยีสต์", "น้ำ", "เกลือ", "น้ำตาล", "เนย"]  
    - "ขนมปัง" + "multigrain" → "ขนมปังมัลติเกรน" → ["แป้งสาลี", "ธัญพืชรวม", "ยีสต์", "น้ำ", "เกลือ", "น้ำตาล", "เนย"]  

    **หมายเหตุ:**  
    - คำบ่งบอกชนิดของขนมปังสามารถอยู่ตำแหน่งใดก็ได้ในลิสต์  
    - ถ้าพบขนมปังชนิดนี้ ให้แทนที่รายการเดิมทั้งหมดด้วยรายการส่วนประกอบขนมปังตามชนิดนั้น
    - ถ้าเจอคำว่า "ขนมปัง" เพียงอย่างเดียว (ไม่มีชนิดต่อท้าย) ให้คงเป็นคำว่า "ขนมปัง" และ **ไม่ต้องแยกส่วนประกอบภายใน**

    
22. ถ้าเจอชื่ออาหารที่เป็นของกินเล่น หรืออาหารที่ผ่านการทอด ย่าง ปิ้ง หรือชุบแป้งทอด เช่น  
    - "หมูสะเต๊ะ"  
    - "ลูกชิ้นทอด"  
    - "ไก่ป๊อป"  
    - "ปอเปี๊ยะทอด"  
    - "ปลาหมึกย่าง"  
    - "ไก่กรอบ"  
    - "นักเก็ตไก่"
    - "มันฝรั่งทอดกรอบ"  
    ให้แยกออกเป็นวัตถุดิบหลักของอาหารนั้น + ส่วนประกอบการปรุงพื้นฐาน เช่น  

    - "หมูสะเต๊ะ" → ["หมู", "เครื่องหมักสะเต๊ะ", "กะทิ", "ขมิ้น", "ผงกะหรี่", "น้ำจิ้มถั่วลิสง"]  
    - "ลูกชิ้นทอด" → ["ลูกชิ้น", "น้ำมันทอด", "ซอสพริก"]  
    - "ไก่ป๊อป" → ["ไก่", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]  
    - "ปอเปี๊ยะทอด" → ["แป้งปอเปี๊ยะ", "วุ้นเส้น", "ผักกะหล่ำ", "แครอท", "น้ำมันทอด"]  
    - "ปลาหมึกย่าง" → ["ปลาหมึก", "น้ำจิ้มซีฟู้ด"]  
    - "นักเก็ตไก่" → ["เนื้อไก่บด", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]
    - "มันฝรั่งทอดกรอบ" → ["มันฝรั่ง", "น้ำมันทอด", "เกลือ"]  

    หมายเหตุ: ถ้าเมนูของทอดหรือของกินเล่นมีชื่อวัตถุดิบเพิ่มเติม เช่น  
    - "ลูกชิ้นปลาทอด" หรือ input เดิมมี ["ลูกชิ้นทอด", "ปลา"]  
    ให้คงวัตถุดิบนั้นไว้ในผลลัพธ์ เช่น  
    - ["ลูกชิ้น", "ปลา", "น้ำมันทอด", "ซอสพริก"]

23. ถ้าเจอชื่ออาหารหรือวัตถุดิบที่เป็น “ของดอง / ของหมักดอง” เช่น  
    - "ผักกาดดอง"  
    - "หน่อไม้ดอง"  
    - "แตงกวาดอง"  
    - "ปลาร้า"  
    - "กิมจิ"  
    - "ไข่เค็ม"  
    - "ปลาเค็ม"  
    - "กุ้งเคยหมัก"  
    - "น้ำปลาหมัก"  

    ให้ทำดังนี้:  
    1. ระบุวัตถุดิบหลักก่อนการดองหรือหมัก เช่น  
        - ผักกาดดอง → "ผักกาด"  
        - หน่อไม้ดอง → "หน่อไม้"  
        - ไข่เค็ม → "ไข่"  
        - ปลาเค็ม → "ปลา"  
    2. เพิ่มส่วนประกอบพื้นฐานของการดองหรือหมัก เช่น  
        - ของดอง → "เกลือ", "น้ำส้มสายชู", "น้ำตาล"  
        - ของหมัก → "เกลือ", "รำข้าว", "น้ำปลา", "ข้าวหมัก", "เครื่องปรุงหมัก"  
    3. ถ้าเป็นของหมักที่เป็นเครื่องปรุงเฉพาะ (เช่น ปลาร้า, กะปิ, น้ำปลา)  
        ให้คงชื่อเฉพาะไว้ เช่น  
    
        - "กะปิ" → ["กะปิ"]  
        - "น้ำปลา" → ["น้ำปลา"]
    *ยกเว้น* เครื่องปรุงเฉพาะเดียว คือ ปลาร้า ให้แยกส่วนประกอบ
        - "ปลาร้า" → ["ปลาร้า"] → ["ปลา", "เกลือ", "รำข้าว"]

ตัวอย่าง:
- "ผักกาดดอง" → ["ผักกาด", "เกลือ", "น้ำส้มสายชู", "น้ำตาล"]
- "หน่อไม้ดอง" → ["หน่อไม้", "เกลือ", "น้ำ"]
- "แตงกวาดอง" → ["แตงกวา", "เกลือ", "น้ำส้มสายชู", "น้ำตาล"]
- "ไข่เค็ม" → ["ไข่", "เกลือ", "น้ำ"]
- "ปลาเค็ม" → ["ปลา", "เกลือ", "แสงแดด"]
- "ปลาร้า" → ["ปลา", "เกลือ", "รำข้าว"]
- "กิมจิ" → ["ผักกาดขาว", "พริกป่นเกาหลี", "กระเทียม", "ขิง", "เกลือ", "น้ำปลา"]

25. ถ้าเจอคำที่เกี่ยวข้องกับ “ซาลาเปา” และมีคำอื่นต่อท้ายที่ระบุไส้ เช่น  
    “ซาลาเปาไส้หมู”, “ซาลาเปาไส้หมูสับ”, “ซาลาเปาไส้ถั่วแดง”, “ซาลาเปาไส้ครีม”  
    ให้แยกส่วนประกอบออกเป็นรายการของวัตถุดิบภายในซาลาเปานั้น

    ตัวอย่าง:
    - "ซาลาเปาไส้หมู" → ["แป้งสาลี", "หมูสับ", "น้ำตาล", "ยีสต์", "ซอสปรุงรส", "น้ำ"]
    - "ซาลาเปาไส้ครีม" → ["แป้งสาลี", "ไข่", "น้ำตาล", "นม", "เนย", "ยีสต์"]
    - "ซาลาเปาไส้ถั่วแดง" → ["แป้งสาลี", "ถั่วแดง", "น้ำตาล", "ยีสต์", "น้ำ"]

    **หมายเหตุ:**  
    - ให้แยกเฉพาะ “ไส้” ที่ระบุในชื่อเท่านั้น ไม่ต้องแตกส่วนประกอบของไส้เพิ่มเติม เช่น “ไส้หมูสับ” ให้คงเป็น “หมูสับ”  
    - ถ้าเจอคำว่า “ซาลาเปา” เพียงอย่างเดียว (ไม่มีคำว่าไส้ต่อท้าย) ให้คงเป็น “ซาลาเปา” โดย **ไม่ต้องแยกส่วนประกอบ**
    - ถ้าพบ “ซาลาเปา” ร่วมกับคำอื่นที่ไม่ใช่ไส้ เช่น “ซาลาเปานึ่ง”, “ซาลาเปาเย็น” ให้ถือว่าเป็น “ซาลาเปา” เช่นเดิม

26. ถ้าเจอคำว่า “ชา” และมีคำต่อท้ายที่ระบุรสหรือผลไม้ เช่น
    - "ชาส้ม", "ชาเขียวมะลิ", "ชามะนาว"
    ให้คงเป็นชื่อเครื่องดื่มเดียว **ไม่ต้องแยกเป็น "ชา" + ผลไม้"**
    - ตัวอย่าง: "ชาส้ม" → ["ชาส้ม"], "ชาเขียวมะลิ" → ["ชาเขียวมะลิ"]

27. ถ้าพบคำขยายที่บ่งบอกขนาดหรือรูปทรงของอาหาร เช่น "ก้อน", "ชิ้นเล็ก", "ชิ้นใหญ่"  
    ให้ลบคำเหล่านี้ออก เหลือเฉพาะชื่ออาหารหรือวัตถุดิบหลัก  
    - ตัวอย่าง:  
      "หมูบดก้อน" → "หมูบด"  
      "เต้าหู้ชิ้นเล็ก" → "เต้าหู้"  
      "เนื้อชิ้นใหญ่" → "เนื้อ"  
    - **ข้อสำคัญ:** คำที่บ่งบอกวิธีทำหรือชนิดของอาหาร เช่น "สับ", "บด", "หมัก", "ทอด", "ย่าง" ให้คงไว้

28. ถ้าเจอคำว่า "ผล" เพียงคำเดียว ให้ **ไม่ตีความเป็น "ผลไม้"** และไม่เพิ่มลงในรายการอาหาร  
- ตัวอย่าง:  
  "ผล" → ถูกลบออก  
  "ผลไม้รวม" → คงไว้ตามปกติเป็นชื่ออาหาร

29. ถ้าเจอชื่ออาหารที่เป็น **เมนูหลักสองเมนูติดกัน** เช่น  
    - "ก๋วยเตี๋ยวราดหน้า"  
    - "ข้าวมันไก่ทอด"  
  ให้ทำดังนี้:  
  1. แยกคำเป็นเมนูหลักแยกกัน เช่น  
     - "ก๋วยเตี๋ยวราดหน้า" → ["ก๋วยเตี๋ยว", "ราดหน้า"]  
     - "ข้าวมันไก่ทอด" → ["ข้าวมันไก่", "ไก่ทอด"]  
  2. แยกส่วนประกอบของแต่ละเมนูตามกฎที่เกี่ยวข้องกับประเภทเมนูนั้น เช่น  
     - "ก๋วยเตี๋ยว" → ["เส้นก๋วยเตี๋ยว", "น้ำซุป", "ผัก"]  
     - "ราดหน้า" → ["เส้นใหญ่", "หมู", "ซอสราดหน้า", "ผัก"]  
     - "ไก่ทอด" → ["ไก่", "แป้งทอดกรอบ", "น้ำมันทอด", "เกลือ", "พริกไทย"]

30. ให้ส่งผลลัพธ์กลับมาในรูปแบบ JSON object เท่านั้น โดยมีโครงสร้างดังนี้:

    - key = ชื่ออาหารหรือคำต้นฉบับแต่ละคำ (จากลิสต์ input)
    - value = list ของส่วนประกอบหรือคำที่ผ่านการแก้ไขแล้ว  
    
    *** เงื่อนไขสำคัญ ***
    - ถ้าในบรรทัด input มีหลายคำ เช่น ['หน่อไม้ดอง', 'กะทิ']  
    ให้แยกเป็นหลาย key โดยแต่ละคำต้องมี list ของตัวเอง เช่น

    {{
        "หน่อไม้ดอง": ["หน่อไม้", "เกลือ", "น้ำ"],
        "กะทิ": ["กะทิ"]
    }}
    - ห้ามรวมคำหลายคำไว้ใน key เดียว เช่น "['หน่อไม้ดอง', 'กะทิ']"
    - ต้องแยกออกเป็น key แต่ละคำเสมอ
    - ให้รูปแบบ JSON ถูกต้องตามมาตรฐาน (ไม่ต้องมีคำอธิบายหรือข้อความอื่นนอกเหนือจาก JSON)

"""



response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.1,
        thinking_config=types.ThinkingConfig(thinking_budget=0)
    ),
)

text = response.text.strip()
print("Raw response:\n", text)


Raw response:
 ```json
{
  "มะระ": ["มะระ"],
  "หมู": ["หมู"],
  "ผักชี": ["ผักชี"],
  "เห็นชอบ": [],
  "กะปิ": ["กะปิ"],
  "สะตอ": ["สะตอ"],
  "ข้าว": ["ข้าว"],
  "ผักบุ้ง": ["ผักบุ้ง"],
  "หมู.คะน้า": ["หมู", "คะน้า"],
  "หอมใหญ่": ["หอมใหญ่"],
  "มะเขือเทศ": ["มะเขือเทศ"],
  "ไข่": ["ไข่"],
  "กาแฟ": ["กาแฟ"],
  "ข้าวเหนียว": ["ข้าวเหนียว"],
  "เนื้อวัว": ["เนื้อวัว"],
  "ใบกะเพรา": ["ใบกะเพรา"],
  "พริก": ["พริก"],
  "กะเพรา": ["ใบกะเพรา"],
  "ไข่ไก่": ["ไข่"],
  "กะทิ": ["กะทิ"],
  "มะเขือพวง": ["มะเขือพวง"],
  "ใบมะกรูด": ["ใบมะกรูด"],
  "ผักกาดดอง": ["ผักกาด", "เกลือ", "น้ำส้มสายชู", "น้ำตาล"],
  "เนื้อไก่": ["เนื้อไก่"],
  "กระเทียม": ["กระเทียม"],
  "หอมแดง": ["หอมแดง"],
  "ข้าวเหนียวสังขยา": ["ข้าวเหนียวสังขยา"],
  "ขิง": ["ขิง"],
  "ต้นหอม": ["ต้นหอม"],
  "กุ้งแห้ง": ["กุ้งแห้ง"],
  "ถั่วฝักยาว": ["ถั่วฝักยาว"],
  "มะม่วง": ["มะม่วง"],
  "ข่า": ["ข่า"],
  "ไข่เจียว": ["ไข่", "น้ำมันทอด", "เกลือ", "พริกไทย"],
  "หน่อไม้": ["หน่อไม้"],
  "ชะอม": ["ชะอม"],
  "ไก่": ["ไก่"],
  "

In [65]:
import json
import re

def extract_json(text: str) -> dict:
    # Remove ```json ``` or ``` ```
    text = re.sub(r"```(?:json)?", "", text)
    text = text.strip("` \n")

    # Extract first JSON object
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in LLM response")

    return json.loads(match.group())
merge_map = extract_json(text)

In [66]:
def normalize_items(items, merge_map):
    result = set()

    for item in items:
        if item in merge_map:
            for mapped in merge_map[item]:
                result.add(mapped)
        else:
            result.add(item)

    return list(result)


In [67]:
df['AIClean_menu'] = df['Cleaned_menu'].apply(
    lambda x: normalize_items(x, merge_map)
)

In [68]:
selected_cols = ["Cleaned_menu", "AIClean_menu"]

df.loc[:50, selected_cols].head(51)

,Cleaned_menu,AIClean_menu
0,"[มะระ, หมู, ผักชี, เห็นชอบ, กะปิ, สะตอ, ข้าว, ...","[มะระ, ข้าว, ผักชี, สะตอ, หมู, กะปิ, ผักบุ้ง]"
1,"[ข้าว, หมู, คะน้า, หอมใหญ่, มะเขือเทศ, ไข่]","[มะเขือเทศ, ข้าว, หอมใหญ่, ไข่, คะน้า, หมู]"
2,"[กาแฟ, ข้าวเหนียว, หมู]","[หมู, ข้าวเหนียว, กาแฟ]"
3,"[ข้าว, เนื้อวัว, ใบกะเพรา, พริก, กะเพรา, ไข่ไก่]","[ข้าว, ใบกะเพรา, ไข่, พริก, เนื้อวัว]"
4,"[ข้าว, เนื้อหมู, กะทิ, มะเขือพวง, ใบมะกรูด, ผั...","[เกลือ, ข้าว, ข้าวเหนียวสังขยา, กระเทียม, ผักก..."
5,"[ข้าว, หมู, ขิง, ต้นหอม, ไข่, กาแฟ]","[ต้นหอม, กาแฟ, ข้าว, ไข่, ขิง, หมู]"
6,"[ข้าว, หมู, กะปิ, กุ้งแห้ง, ถั่วฝักยาว, มะม่วง...","[มะม่วง, ข้าว, กุ้งแห้ง, ถั่วฝักยาว, ไข่, พริก..."
7,"[ข้าว, เนื้อวัว, พริก, ข่า, ใบมะกรูด, ไข่เจียว...","[ข่า, น้ำมันทอด, ข้าว, เกลือ, สะตอ, พริกไทย, เ..."
8,"[ข้าว, ไก่, พริก, กะปิ, กระเทียม, เนื้อวัว, ข่า]","[ข่า, ข้าว, กระเทียม, พริก, เนื้อวัว, กะปิ, ไก่]"
9,"[เนื้อวัว, พริกไทย, มันฝรั่ง, สลัดผัก, (กระหล่...","[(กระหล่ำปลี, แครอท, โค้ก, พริกไทย, ข้าวโพด, เ..."


In [69]:
replace_dict = {

    "เส้นมาม่า": "มาม่า",
    "บะหมี่กึ่งสำเร็จรูป" : "มาม่า",
    
    "มะนาว" : "น้ำมะนาว",

    "ไข่ดาวไม่สุก" : "ไข่ดาว",

    "ขาหมู": "ข้าวขาหมู",
    "กะเพรา": "ใบกะเพรา",
    "คะน้า": "ผักคะน้า",
    
    "กะทิ" : "น้ำกะทิ",

    "ฟักทองญี่ปุ่น" : "ฟักทอง",
    "ถั่วแระญี่ปุ่น" : "ถั่วแระ",
    'ต้นหอมญี่ปุ่น' : 'ต้นหอม',

    "ข้าวญี่ปุ่น" : "ข้าว",
    'ซูชิ' : "ข้าวปั้นญี่ปุ่น",

    "ขนมปังกรอบ" : "ขนมปัง",
    "ขนมปังกรอบไร้กลูเตน" : "ขนมปัง",
    "ขนมปังฝรั่งเศส" : "ขนมปัง",

    "บอดี้คีย์" : "บอดีคีย์",

}

df["FinalCleaned_menu"] = df["AIClean_menu"].apply(
    lambda lst: [replace_dict.get(item, item) for item in lst]
)

In [70]:
def split_food_name(name):
    if name == "ข้าวมันไก่":
        return ['ข้าว', 'ไก่', 'น้ำจิ้มเต้าเจี้ยว', 'น้ำซุป']
    elif name == "ข้าวคลุกกะปิ":
        return ["ข้าว", "กะปิ", "หมูหวาน", "กุ้งแห้ง", "ไข่เจียว", "หอมแดง", "ถั่วฝักยาว", "มะม่วง"]
    elif name == "น้ำสลัดงาคั่วญี่ปุ่น":
        return ["น้ำสลัด", "งา"]
    elif name == 'อกเป็ดรมควัน' :
        return ['เป็ด', 'ซอส', 'ผัก']
    elif name == 'น้ำสลัดงาญี่ปุ่น':
        return ["น้ำสลัด", "งา"]
    elif name == 'ข้าวปั้นญี่ปุ่น' : 
        return ['ข้าว', 'น้ำส้มซูชิ', 'สาหร่าย', 'ปลา', 'ไข่', 'ผัก']
    elif name == 'น้ำสลัด':
        return ['น้ำมัน', 'น้ำส้มสายชู', 'น้ำตาล', 'เครื่องเทศ']
    elif name == 'น้ำสลัดซีฟู้ด': 
        return ['น้ำมัน', 'น้ำมะนาว', 'เครื่องเทศ', 'น้ำมันหอย', 'ซอสปลา']
    elif name == 'น้ำสลัดซีซาร์':
        return ['มายองเนส', 'น้ำมันพืช', 'น้ำมะนาว', 'พาร์เมซานชีส', 'กระเทียม', 'ซอสวูสเตอร์เชียร์', 'พริกไทย']
    elif name == 'น้ำสลัดครีม': 
        return ['มายองเนส', 'ครีมข้น', 'น้ำตาล', 'น้ำส้มสายชู', 'เกลือ', 'พริกไทย', 'กระเทียม']
    elif name == 'ลูกชิ้นปลา': 
        return ['ปลา', 'แป้ง', 'ไข่', 'น้ำตาล', 'เกลือ', 'เครื่องปรุงรส', 'น้ำ', "น้ำส้มสายชู"]

    else:
        return [name]

df["FinalCleaned_menu"] = df["FinalCleaned_menu"].apply(
    lambda foods: sum([split_food_name(f) for f in foods], [])
)


In [ ]:
#Delete Fodmap ingredient
def remove_words(food_list, remove_list):
    if isinstance(food_list, list):
        return [str(item) for item in food_list if item not in remove_list]
    else:
        return []  

words_to_remove = ['หมู',"เนื้อหมู", "หมูสับ","ลูกชิ้นหมู", "พริก", "ของว่าง", 'อาหารเสริมโปรตีนชง', "โปรตีน", "โปรตีนผง", 'โปรตีนชง', 'อาหารเสริมโปรตีนชง',"บอดีคีย์"]

words_to_remove_egg = ["ไข่ขาว", "ไข่แดง", "ไข่"]

word_to_remove_fish = ['ปลา','ปลาทู', "เนื้อปลา", 'ปลาสลิด', 'ปลาหมึก']

words_to_remove_meat = ['เนื้อวัว', 'เนื้อ', 'เนื้อกุ้ง', 'ไก่', 'เนื้อไก่', 'เครื่องในวัว', 'กุ้ง', "เครื่องใน"]

words_to_remove_carb = ['ข้าว', "ข้าวสวย", "แป้งทอดกรอบ", "ข้าวต้ม"]

word_to_remove_sauce = ["น้ำตาล", "พริกไทย", "น้ำ", "น้ำเปล่า", "เกลือ", "น้ำปลา", "น้ำมันทอด", "น้ำมัน", "เครื่องปรุงรส", "น้ำส้มสายชู"]
all_remove_words = (
    words_to_remove
    + words_to_remove_meat
    + words_to_remove_carb
    + words_to_remove_egg
    + word_to_remove_fish
    + word_to_remove_sauce
)

df["FinalCleaned_menu"] = df["FinalCleaned_menu"].apply(lambda x: remove_words(x, all_remove_words))



In [72]:
selected_cols = ["Cleaned_menu", "AIClean_menu", "FinalCleaned_menu"]

df.loc[:50, selected_cols].head(51)

,Cleaned_menu,AIClean_menu,FinalCleaned_menu
0,"[มะระ, หมู, ผักชี, เห็นชอบ, กะปิ, สะตอ, ข้าว, ...","[มะระ, ข้าว, ผักชี, สะตอ, หมู, กะปิ, ผักบุ้ง]","[มะระ, ผักชี, สะตอ, กะปิ, ผักบุ้ง]"
1,"[ข้าว, หมู, คะน้า, หอมใหญ่, มะเขือเทศ, ไข่]","[มะเขือเทศ, ข้าว, หอมใหญ่, ไข่, คะน้า, หมู]","[มะเขือเทศ, หอมใหญ่, ผักคะน้า]"
2,"[กาแฟ, ข้าวเหนียว, หมู]","[หมู, ข้าวเหนียว, กาแฟ]","[ข้าวเหนียว, กาแฟ]"
3,"[ข้าว, เนื้อวัว, ใบกะเพรา, พริก, กะเพรา, ไข่ไก่]","[ข้าว, ใบกะเพรา, ไข่, พริก, เนื้อวัว]",[ใบกะเพรา]
4,"[ข้าว, เนื้อหมู, กะทิ, มะเขือพวง, ใบมะกรูด, ผั...","[เกลือ, ข้าว, ข้าวเหนียวสังขยา, กระเทียม, ผักก...","[ข้าวเหนียวสังขยา, กระเทียม, ผักกาด, หอมแดง, ม..."
5,"[ข้าว, หมู, ขิง, ต้นหอม, ไข่, กาแฟ]","[ต้นหอม, กาแฟ, ข้าว, ไข่, ขิง, หมู]","[ต้นหอม, กาแฟ, ขิง]"
6,"[ข้าว, หมู, กะปิ, กุ้งแห้ง, ถั่วฝักยาว, มะม่วง...","[มะม่วง, ข้าว, กุ้งแห้ง, ถั่วฝักยาว, ไข่, พริก...","[มะม่วง, กุ้งแห้ง, ถั่วฝักยาว, กะปิ]"
7,"[ข้าว, เนื้อวัว, พริก, ข่า, ใบมะกรูด, ไข่เจียว...","[ข่า, น้ำมันทอด, ข้าว, เกลือ, สะตอ, พริกไทย, เ...","[ข่า, สะตอ, ชะอม, หน่อไม้, น้ำกะทิ, ใบมะกรูด]"
8,"[ข้าว, ไก่, พริก, กะปิ, กระเทียม, เนื้อวัว, ข่า]","[ข่า, ข้าว, กระเทียม, พริก, เนื้อวัว, กะปิ, ไก่]","[ข่า, กระเทียม, กะปิ]"
9,"[เนื้อวัว, พริกไทย, มันฝรั่ง, สลัดผัก, (กระหล่...","[(กระหล่ำปลี, แครอท, โค้ก, พริกไทย, ข้าวโพด, เ...","[(กระหล่ำปลี, แครอท, โค้ก, ข้าวโพด, ผักสลัด, เ..."
